# Feature engineering
Every history feature uses **earlier races only**, ordered by exact race date.

In [1]:
import pandas as pd

df = pd.read_csv("data/all_marathons_final.csv", low_memory=False)
df = df.drop(columns=["index", "Gun Time", "Place", "Sex Place", "Div Place"])  # leak the target
df["rid"] = range(len(df))
df.shape

(3343216, 11)

## Remove duplicated race
NYC 2006 is an exact copy of NYC 2007 (winner Martin Lel, 2:09:04 = the 2007 race).

In [2]:
dup = (df["Marathon"] == "NYC") & (df["Year"] == 2006)
df = df[~dup].reset_index(drop=True)
df["rid"] = range(len(df))
dup.sum()

np.int64(38555)

## Race date
Exact date of every edition (checked online: Wikipedia winners tables and MarathonGuide race pages).
All Sundays except Grandma's (Saturday) and LA 2009 (Memorial Day Monday). LA moved to May 2009 and Feb 2016.

In [3]:
DATES = {
    "Houston": ["2000-01-16", "2001-01-14", "2002-01-20", "2003-01-19", "2004-01-18", "2005-01-16", "2006-01-15",
                "2007-01-14", "2008-01-13", "2009-01-18", "2010-01-17", "2011-01-30", "2012-01-15", "2013-01-13",
                "2014-01-19", "2015-01-18", "2016-01-17", "2017-01-15", "2018-01-14", "2019-01-20"],
    "LA": ["2000-03-05", "2001-03-04", "2002-03-03", "2003-03-02", "2004-03-07", "2005-03-06", "2006-03-19",
           "2007-03-04", "2008-03-02", "2009-05-25", "2010-03-21", "2011-03-20", "2012-03-18", "2013-03-17",
           "2014-03-09", "2015-03-15", "2016-02-14", "2017-03-19", "2018-03-18", "2019-03-24"],
    "Grandmas": ["2000-06-17", "2001-06-16", "2002-06-22", "2003-06-21", "2004-06-19", "2005-06-18", "2007-06-16",
                 "2008-06-21", "2009-06-20", "2010-06-19", "2011-06-18", "2012-06-16", "2013-06-22", "2014-06-21",
                 "2015-06-20", "2016-06-18", "2017-06-17", "2018-06-16", "2019-06-22"],
    "Twin_Cities": ["2000-10-08", "2001-10-07", "2002-09-29", "2004-10-03", "2006-10-01", "2007-10-07", "2008-10-05",
                    "2009-10-04", "2010-10-03", "2011-10-02", "2012-10-07", "2013-10-06", "2014-10-05", "2015-10-04",
                    "2016-10-09", "2017-10-01", "2018-10-07", "2019-10-06"],
    "Chicago": ["2000-10-22", "2001-10-07", "2002-10-13", "2003-10-12", "2004-10-10", "2005-10-09", "2006-10-22",
                "2007-10-07", "2008-10-12", "2009-10-11", "2010-10-10", "2011-10-09", "2012-10-07", "2013-10-13",
                "2014-10-12", "2015-10-11", "2016-10-09", "2017-10-08", "2018-10-07", "2019-10-13"],
    "MarineCorps": ["2000-10-22", "2001-10-28", "2002-10-27", "2003-10-26", "2004-10-31", "2005-10-30", "2006-10-29",
                    "2007-10-28", "2008-10-26", "2009-10-25", "2010-10-31", "2011-10-30", "2012-10-28", "2013-10-27",
                    "2014-10-26", "2015-10-25", "2016-10-30", "2017-10-22", "2018-10-28", "2019-10-27"],
    "NYC": ["2000-11-05", "2001-11-04", "2002-11-03", "2003-11-02", "2004-11-07", "2005-11-06", "2007-11-04",
            "2008-11-02", "2009-11-01", "2010-11-07", "2011-11-06", "2013-11-03", "2014-11-02", "2015-11-01",
            "2016-11-06", "2017-11-05", "2018-11-04", "2019-11-03"],  # 2006 dropped above, 2012 cancelled (Sandy)
    "Philadelphia": ["2000-11-19", "2001-11-18", "2002-11-24", "2003-11-23", "2004-11-21", "2005-11-20", "2006-11-19",
                     "2007-11-18", "2008-11-23", "2009-11-22", "2010-11-21", "2011-11-20", "2012-11-18", "2013-11-17",
                     "2014-11-23", "2015-11-22", "2016-11-20", "2017-11-19", "2018-11-18", "2019-11-24"],
    "California_International": ["2000-12-03", "2001-12-02", "2002-12-08", "2003-12-07", "2004-12-05", "2005-12-04",
                                 "2006-12-03", "2007-12-02", "2008-12-07", "2009-12-06", "2010-12-05", "2011-12-04",
                                 "2012-12-02", "2013-12-08", "2014-12-07", "2015-12-06", "2016-12-04", "2017-12-03",
                                 "2018-12-02", "2019-12-08"],
    "Honolulu": ["2000-12-10", "2001-12-09", "2002-12-08", "2003-12-14", "2004-12-12", "2005-12-11", "2006-12-10",
                 "2007-12-09", "2008-12-14", "2009-12-13", "2010-12-12", "2011-12-11", "2012-12-09", "2013-12-08",
                 "2014-12-14", "2015-12-13", "2016-12-11", "2017-12-10", "2018-12-09", "2019-12-08"],
}
dates = pd.DataFrame([(m, d) for m, ds in DATES.items() for d in ds], columns=["Marathon", "race_date"])
dates["race_date"] = pd.to_datetime(dates["race_date"])
dates["Year"] = dates["race_date"].dt.year
df = df.merge(dates, on=["Marathon", "Year"], how="left")
assert df["race_date"].notna().all()
df["Month"] = df["race_date"].dt.month
df.groupby("Marathon")["race_date"].agg(["min", "max", "nunique"])

,min,max,nunique
Marathon,,,
California_International,2000-12-03,2019-12-08,20
Chicago,2000-10-22,2019-10-13,20
Grandmas,2000-06-17,2019-06-22,19
Honolulu,2000-12-10,2019-12-08,20
Houston,2000-01-16,2019-01-20,20
LA,2000-03-05,2019-03-24,20
MarineCorps,2000-10-22,2019-10-27,20
NYC,2000-11-05,2019-11-03,18
Philadelphia,2000-11-19,2019-11-24,20


## Target

In [4]:
df["Slow"] = (df["Time"] > 360).astype(int)
df["Slow"].mean()

np.float64(0.14056751963363262)

## Runner linking
Same runner = same name + sex + overlapping birth-year range.

In [5]:
df["name_sex"] = df["Name"].str.upper().str.split().str.join(" ") + "|" + df["Sex"]
placeholder = df["Name"].str.upper().str.contains("UNKNOWN RUNNER|RUNNER UNKNOWN", na=True)
df.loc[placeholder, "name_sex"] = None
placeholder.sum()

np.int64(235)

Exact age: birth year is Year − Age, or one less if the birthday is after race day.

In [6]:
df["by_lo"] = df["Year"] - df["Age"] - 1
df["by_hi"] = df["Year"] - df["Age"]

Fallback when age is missing: 5-year age group only (18-34, 17U, 80+ are too wide).

In [7]:
bounds = df["Agegroup"].str.extract(r"^(\d+)-(\d+)$").astype(float)
use_group = df["Age"].isna() & ((bounds[1] - bounds[0]) <= 4)
df.loc[use_group, "by_lo"] = df["Year"] - bounds[1] - 1
df.loc[use_group, "by_hi"] = df["Year"] - bounds[0]
use_group.sum()

np.int64(301959)

No fallback for names shared by several people (exact birth years > 2 apart).

In [8]:
aged = df[df["Age"].notna()].groupby("name_sex")["by_hi"]
shared_name = (aged.max() - aged.min()) > 2
ambiguous = use_group & df["name_sex"].map(shared_name).fillna(False).astype(bool)
df.loc[ambiguous, ["by_lo", "by_hi"]] = None

df["linked_by_agegroup"] = (use_group & ~ambiguous).astype(int)
df["linkable"] = df["name_sex"].notna() & df["by_lo"].notna()
ambiguous.sum(), df["linkable"].mean()

(np.int64(36369), np.float64(0.9384587405485767))

## Race-level stats (per marathon and year)

In [9]:
race = df.groupby(["Marathon", "Year"]).agg(
    race_median=("Time", "median"),
    race_slow_rate=("Slow", "mean"),
    race_size=("Time", "size"),
).reset_index()
race["year_median"] = race.groupby("Year")["race_median"].transform("median")
race["difficulty"] = race["race_median"] / race["year_median"]  # >1 = slower course than a typical race that year
race.head()

,Marathon,Year,race_median,race_slow_rate,race_size,year_median,difficulty
0,California_International,2000,242.066667,0.016642,2704,257.900000,0.938607
1,California_International,2001,255.416667,0.015768,2727,259.583333,0.983949
2,California_International,2002,238.683333,0.007660,2611,259.316667,0.920432
3,California_International,2003,248.950000,0.019454,2930,264.233333,0.942160
4,California_International,2004,242.733333,0.018919,2960,264.962500,0.916104


## Course features (previous years only)
Expanding mean over earlier editions of the same race, shifted so the current year is excluded.

In [10]:
race = race.sort_values(["Marathon", "Year"])
by_race = race.groupby("Marathon")
race["course_difficulty"] = by_race["difficulty"].transform(lambda s: s.shift().expanding().mean())
race["course_slow_rate"] = by_race["race_slow_rate"].transform(lambda s: s.shift().expanding().mean())
race["course_prev_size"] = by_race["race_size"].shift()

df = df.merge(race[["Marathon", "Year", "race_median", "difficulty", "course_difficulty", "course_slow_rate", "course_prev_size"]],
              on=["Marathon", "Year"], how="left")
race[race["Marathon"] == "MarineCorps"][["Year", "difficulty", "course_difficulty", "course_slow_rate"]].tail()

,Year,difficulty,course_difficulty,course_slow_rate
134,2015,1.078145,1.061947,0.086190
135,2016,1.107195,1.062960,0.088886
136,2017,1.096282,1.065562,0.093906
137,2018,1.094057,1.067268,0.098167
138,2019,1.164212,1.068678,0.099613


## Relative and course-adjusted time
Use the current race's results → only used for **past** races below (never the current one).

In [11]:
df["rel_time"] = df["Time"] / df["race_median"]   # vs the field that day
df["adj_time"] = df["Time"] / df["difficulty"]     # time on an "average" course

## Pair each race with the same runner's earlier races

In [12]:
link = df.loc[df["linkable"], ["rid", "name_sex", "by_lo", "by_hi", "race_date", "Marathon", "linked_by_agegroup"]]
hist = df.loc[df["linkable"], ["name_sex", "by_lo", "by_hi", "race_date", "Marathon", "Time", "rel_time", "adj_time",
                               "difficulty", "Slow", "linked_by_agegroup"]]
pairs = link.merge(hist, on="name_sex", suffixes=("", "_prev"))
pairs = pairs[(pairs["race_date_prev"] < pairs["race_date"])  # same day (Chicago & MCM, 22 Oct 2000) not counted
              & (pairs["by_lo_prev"] <= pairs["by_hi"]) & (pairs["by_lo"] <= pairs["by_hi_prev"])]  # ranges overlap
pairs["same_race"] = pairs["Marathon_prev"] == pairs["Marathon"]
pairs["via_agegroup"] = pairs["linked_by_agegroup"] | pairs["linked_by_agegroup_prev"]
len(pairs), pairs["via_agegroup"].mean()

(3679911, np.float64(0.1194072356641234))

## Runner features (all previous races)

In [13]:
g = pairs.groupby("rid")
runner_feats = pd.DataFrame({
    "prev_races": g.size(),
    "prev_median_time": g["Time"].median(),
    "prev_best_time": g["Time"].min(),
    "prev_slow_rate": g["Slow"].mean(),
    "prev_median_rel_time": g["rel_time"].median(),
    "prev_median_adj_time": g["adj_time"].median(),
    "prev_best_adj_time": g["adj_time"].min(),
    "prev_median_course_difficulty": g["difficulty"].median(),
    "prev_distinct_races": g["Marathon_prev"].nunique(),
    "days_since_last": (g["race_date"].first() - g["race_date_prev"].max()).dt.days,
    "prev_agegroup_links": g["via_agegroup"].sum(),
})

last = pairs.sort_values("race_date_prev").groupby("rid").tail(1).set_index("rid")
runner_feats["prev_last_time"] = last["Time"]

## Runner features (previous editions of the same race)

In [14]:
same = pairs[pairs["same_race"]].groupby("rid")
runner_feats["prev_same_races"] = same.size()
runner_feats["prev_same_median_time"] = same["Time"].median()

## Merge back
Linkable runners with no earlier race get 0 counts. Times stay NaN.

In [15]:
df = df.merge(runner_feats, left_on="rid", right_index=True, how="left")
has_id = df["linkable"]
for col in ["prev_races", "prev_same_races", "prev_agegroup_links", "prev_distinct_races"]:
    df.loc[has_id, col] = df.loc[has_id, col].fillna(0)
df["first_marathon"] = (df["prev_races"] == 0).astype("Int64").where(has_id)

## Expected time on this course
Course-adjusted past median × this course's difficulty (earlier editions only).

In [16]:
df["expected_time"] = df["prev_median_adj_time"] * df["course_difficulty"]
df[["expected_time", "Time"]].corr().round(3)

,expected_time,Time
expected_time,1.0,0.8
Time,0.8,1.0


## Ratio: current time vs own past median
Uses the current time → **analysis only, not a model feature**.

In [17]:
df["time_vs_prev_median"] = df["Time"] / df["prev_median_time"]
df["time_vs_prev_median"].describe()

count    1.268744e+06
mean     1.017876e+00
std      1.428365e-01
min      2.900870e-01
25%      9.383901e-01
50%      1.002678e+00
75%      1.079715e+00
max      4.040943e+00
Name: time_vs_prev_median, dtype: float64

## Check

In [18]:
cols = ["prev_races", "prev_same_races", "prev_median_time", "prev_best_time", "prev_last_time",
        "prev_slow_rate", "prev_median_rel_time", "prev_median_adj_time", "prev_best_adj_time",
        "prev_median_course_difficulty", "prev_distinct_races", "expected_time", "days_since_last", "prev_agegroup_links", "prev_same_median_time",
        "course_difficulty", "course_slow_rate", "course_prev_size"]
df[cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
prev_races,3101288.0,1.19,2.41,0.00,0.00,0.00,1.00,63.00
prev_same_races,3101288.0,0.75,1.77,0.00,0.00,0.00,1.00,31.00
prev_median_time,1268744.0,277.42,61.28,123.75,235.04,268.33,308.82,970.55
prev_best_time,1268744.0,266.22,60.28,123.75,224.58,256.77,296.58,970.55
prev_last_time,1268744.0,279.04,64.68,123.75,233.83,268.98,312.93,975.95
prev_slow_rate,1268744.0,0.10,0.28,0.00,0.00,0.00,0.00,1.00
prev_median_rel_time,1268744.0,0.99,0.18,0.39,0.86,0.97,1.10,2.73
prev_median_adj_time,1268744.0,262.60,49.26,101.87,228.19,258.37,292.64,738.68
prev_best_adj_time,1268744.0,251.60,50.04,101.21,216.45,246.34,281.93,738.68
prev_median_course_difficulty,1268744.0,1.06,0.11,0.89,0.99,1.01,1.09,1.40


In [19]:
df.groupby("Year")["prev_races"].apply(lambda s: (s > 0).mean()).round(3)

Year
2000    0.025
2001    0.263
2002    0.322
2003    0.371
2004    0.318
2005    0.366
2006    0.369
2007    0.393
2008    0.398
2009    0.384
2010    0.413
2011    0.417
2012    0.455
2013    0.433
2014    0.458
2015    0.422
2016    0.479
2017    0.496
2018    0.371
2019    0.364
Name: prev_races, dtype: float64

Share of runners we can link, per race and year.

In [20]:
df.pivot_table(index="Marathon", columns="Year", values="linkable", aggfunc="mean").round(2)

Year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
Marathon,,,,,,,,,,,,,,,,,,,,
California_International,0.97,0.98,1.0,0.96,1.00,1.00,1.0,1.0,1.00,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00
Chicago,1.00,1.00,1.0,1.00,1.00,1.00,1.0,1.0,1.00,1.00,1.00,1.00,1.00,1.0,1.00,0.47,1.00,1.00,0.66,0.55
Grandmas,1.00,1.00,1.0,1.00,0.43,1.00,NaN,1.0,1.00,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00
Honolulu,1.00,1.00,1.0,1.00,0.50,0.49,1.0,1.0,0.53,0.53,0.54,0.57,0.56,0.6,0.62,0.59,0.64,0.65,0.64,0.65
Houston,1.00,1.00,1.0,1.00,1.00,1.00,1.0,1.0,1.00,1.00,1.00,0.51,1.00,1.0,1.00,1.00,1.00,1.00,1.00,0.53
LA,1.00,0.89,1.0,1.00,1.00,1.00,1.0,1.0,1.00,1.00,0.99,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00
MarineCorps,1.00,1.00,1.0,1.00,1.00,1.00,1.0,1.0,1.00,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.00,0.60
NYC,1.00,1.00,1.0,1.00,1.00,1.00,NaN,1.0,1.00,1.00,1.00,1.00,NaN,1.0,1.00,1.00,1.00,1.00,1.00,1.00
Philadelphia,1.00,1.00,1.0,1.00,1.00,1.00,1.0,1.0,1.00,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00


Do age-group links look different? Compare rows linked by exact age vs age group.

In [21]:
df[df["prev_races"] > 0].groupby("linked_by_agegroup")[["prev_races", "prev_median_time", "prev_slow_rate", "Slow"]].mean().round(3)

,prev_races,prev_median_time,prev_slow_rate,Slow
linked_by_agegroup,,,,
0,2.839,273.301,0.082,0.094
1,3.570,322.198,0.291,0.332


Early years look like first-timers because history starts in 2000 → train on later years.

In [22]:
df[cols + ["Slow"]].corr()["Slow"].drop("Slow").sort_values().round(3)

prev_distinct_races             -0.058
course_prev_size                -0.057
prev_races                      -0.004
days_since_last                  0.008
prev_same_races                  0.043
prev_agegroup_links              0.079
prev_median_course_difficulty    0.356
prev_best_adj_time               0.389
course_difficulty                0.418
course_slow_rate                 0.427
prev_median_rel_time             0.433
prev_median_adj_time             0.434
prev_best_time                   0.529
expected_time                    0.565
prev_median_time                 0.565
prev_last_time                   0.571
prev_same_median_time            0.604
prev_slow_rate                   0.610
Name: Slow, dtype: float64

## Save

In [23]:
out = df.drop(columns=["rid", "race_date", "name_sex", "by_lo", "by_hi", "Name", "Country", "data", "Sex1", "race_median", "rel_time",
                        "difficulty", "adj_time"])  # use the current race's results
out.to_csv("data/marathons_features.csv", index=False)
out.shape

(3304661, 30)